# QSAR Tuned Model — XGBoost

Evaluate the tuned XGBoost model against the notebook baseline, with feature importance analysis.

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load and curate data

Replicates the curation pipeline from `data_curation_rdkit.ipynb` on the KNIME source CSV.

In [ ]:
df_raw = pd.read_csv('data/the_final_data_base_in_KNIME.csv')
print(f'Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} cols')

In [ ]:
# Drop metadata rows
meta = df_raw['CAS_str'].astype(str).str.contains('Meta', na=False)
df = df_raw[~meta].reset_index(drop=True)
print(f'After Meta drop: {df.shape[0]} rows')

In [ ]:
# Parse SMILES
def parse_smiles(s):
    try: return Chem.MolFromSmiles(s)
    except: return None
df['mol'] = df['SMILES'].apply(parse_smiles)
df = df.dropna(subset=['mol']).reset_index(drop=True)
print(f'Valid SMILES: {len(df)}')

In [ ]:
# Remove salts
from rdkit.Chem import MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize

def remove_salt(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    if len(frags) == 1: return mol
    largest = max(frags, key=lambda m: m.GetNumAtoms())
    try:
        Chem.SanitizeMol(largest)
        return largest
    except: return mol

df['mol'] = df['mol'].apply(remove_salt)

# Neutralise charges
uncharger = rdMolStandardize.Uncharger()
df['mol'] = df['mol'].apply(uncharger.uncharge)

# Standardise (normalise + tautomer canonicalize)
normalizer = rdMolStandardize.Normalizer()
tautomerizer = rdMolStandardize.TautomerEnumerator()

def standardize(mol):
    try:
        mol = normalizer.normalize(mol)
        mol = tautomerizer.Canonicalize(mol)
    except: pass
    return mol

df['mol'] = df['mol'].apply(standardize)
print('Molecules standardized')

In [ ]:
# Filter inorganics
organic_set = {'H','B','C','N','O','F','Si','P','S','Cl','Se','Br','I'}
def is_inorganic(mol):
    has_c = any(at.GetAtomicNum() == 6 for at in mol.GetAtoms())
    if not has_c: return True
    for at in mol.GetAtoms():
        if at.GetSymbol() not in organic_set: return True
    return False

df['inorganic'] = df['mol'].apply(is_inorganic)
n_inorg = df['inorganic'].sum()
df = df[~df['inorganic']].reset_index(drop=True)
print(f'Removed {n_inorg} inorganics')
print(f'Final: {df.shape[0]} molecules, {df.shape[1]} columns')

## 2. Feature engineering

Morgan fingerprints (radius=2, 2048 bits) + 119 RDKit descriptors, followed by variance and correlation filtering.

In [ ]:
meta_cols = ['CAS_str', 'Chemical_name', 'SMILES', 'mlecule_H_final',
             'SMILES_curated', 'mol', 'inorganic']
target_col = 'log_LC50'
desc_cols = [c for c in df.columns if c not in meta_cols + [target_col, 'n_measurements']]

X_desc = df[desc_cols].values
y = df[target_col].values
print(f'Descriptors: {X_desc.shape[1]}')

In [ ]:
mols = df['mol'].tolist()
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
X_fp = np.array(fps)
print(f'Fingerprints: {X_fp.shape[1]}')

X_combined = np.hstack([X_desc, X_fp])
all_feature_names = desc_cols + [f'FP_{i}' for i in range(X_fp.shape[1])]
print(f'Combined features: {X_combined.shape[1]}')

In [ ]:
# Variance threshold
var_thresh = VarianceThreshold(threshold=0.01)
X_var = var_thresh.fit_transform(X_combined)
kept_mask = var_thresh.get_support()
kept_names = [all_feature_names[i] for i in range(len(all_feature_names)) if kept_mask[i]]
print(f'After variance filter: {X_var.shape[1]}')

# Correlation filter
corr_matrix = pd.DataFrame(X_var).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
keep_idx = [i for i in range(X_var.shape[1]) if i not in to_drop]
X_final = X_var[:, keep_idx]
final_names = [kept_names[i] for i in keep_idx]
print(f'After correlation filter: {X_final.shape[1]}')
print(f'Final features: {X_final.shape[1]}')

## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 4. Baseline model

Notebook-default XGBoost for comparison.

In [ ]:
baseline = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    random_state=42, n_jobs=-1, verbosity=0
)
baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)
base_r2 = r2_score(y_test, y_pred_base)
base_rmse = np.sqrt(mean_squared_error(y_test, y_pred_base))
base_mae = mean_absolute_error(y_test, y_pred_base)

y_train_base = baseline.predict(X_train)
base_gap = r2_score(y_train, y_train_base) - base_r2

print(f'Baseline XGBoost:')
print(f'  Test  R²={base_r2:.4f}  RMSE={base_rmse:.4f}  MAE={base_mae:.4f}')
print(f'  Train R²={r2_score(y_train, y_train_base):.4f}  Gap={base_gap:.4f}')

## 5. Tuned model

Load the best model from hyperparameter tuning.

In [ ]:
with open('models/XGBoost_tuned_final.pkl', 'rb') as f:
    tuned = pickle.load(f)

print('Tuned model parameters:')
for k, v in tuned.get_params().items():
    print(f'  {k}: {v}')

In [ ]:
y_pred_tuned = tuned.predict(X_test)
tuned_r2 = r2_score(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
tuned_mae = mean_absolute_error(y_test, y_pred_tuned)

y_train_tuned = tuned.predict(X_train)
tuned_gap = r2_score(y_train, y_train_tuned) - tuned_r2

print(f'Tuned XGBoost:')
print(f'  Test  R²={tuned_r2:.4f}  RMSE={tuned_rmse:.4f}  MAE={tuned_mae:.4f}')
print(f'  Train R²={r2_score(y_train, y_train_tuned):.4f}  Gap={tuned_gap:.4f}')

## 6. Performance comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['R²', 'RMSE', 'MAE', 'Train R²', 'Gap'],
    'Baseline': [base_r2, base_rmse, base_mae, r2_score(y_train, y_train_base), base_gap],
    'Tuned': [tuned_r2, tuned_rmse, tuned_mae, r2_score(y_train, y_train_tuned), tuned_gap],
    'Δ': [tuned_r2 - base_r2, tuned_rmse - base_rmse, tuned_mae - base_mae,
          r2_score(y_train, y_train_tuned) - r2_score(y_train, y_train_base),
          tuned_gap - base_gap],
})
print(comparison.to_string(index=False))

## 7. Parity plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Parity
axes[0].scatter(y_test, y_pred_tuned, alpha=0.5, edgecolors='none', label='Tuned')
axes[0].scatter(y_test, y_pred_base, alpha=0.3, edgecolors='none', label='Baseline')
lims = [min(y_test.min(), y_pred_tuned.min(), y_pred_base.min()),
        max(y_test.max(), y_pred_tuned.max(), y_pred_base.max())]
axes[0].plot(lims, lims, 'k--', lw=1)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual log_LC50')
axes[0].set_ylabel('Predicted log_LC50')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()
axes[0].set_aspect('equal')

# Residuals
residuals = y_test - y_pred_tuned
axes[1].scatter(y_pred_tuned, residuals, alpha=0.5, edgecolors='none')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted log_LC50')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted (Tuned)')

plt.tight_layout()
plt.savefig('figures/xgboost_tuned_parity.png', dpi=150)
plt.show()

## 8. Feature importance (top 20)

In [ ]:
importances = tuned.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': final_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print('Top 20 features:')
print(feat_imp.head(20).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top20 = feat_imp.head(20)
ax.barh(range(len(top20)), top20['Importance'].values, align='center')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['Feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 20 Features — Tuned XGBoost')
plt.tight_layout()
plt.savefig('figures/xgboost_tuned_feature_importance.png', dpi=150)
plt.show()

## 9. Summary

The tuned XGBoost achieves a modest improvement over the notebook baseline. The persistent train-test gap (~0.46) and plateau at R²~0.50 suggest the dataset is near its predictability ceiling with these features. Likely sources:
- **Experimental noise** in aggregated log_LC50 measurements across different species and protocols
- **Limited feature space** — 511 Morgan fingerprint bits + RDKit 2D descriptors may not capture 3D/electronic effects
- **Further gains** would require new features (e.g., 3D conformer descriptors, quantum-chemical properties) or metadata-guided stratification (e.g., species-specific models)